In [1]:
"""
A simple multi-file aggregate interface for xarray.
"""

from __future__ import annotations

import json
import os
import re
import textwrap
from functools import lru_cache
from pathlib import Path

import attrs
import numpy as np
import pandas as pd
import pint
import xarray as xr

ureg = pint.get_application_registry()


@attrs.define(repr=False, eq=False)
class Aggregate:
    """
    This class represents an aggregate of xarray datasets sharing a common
    dimension. The common dimension is split into contiguous chunks, each
    assigned to one of the aggregated datasets.
    """

    _dirpath: Path
    _index: pd.DataFrame
    _metadata: dict = attrs.field(factory=dict)
    _bins: dict[str, np.ndarray] = attrs.field(factory=dict)

    def __attrs_post_init__(self):
        # Build bin set for all coordinates

        # Parse field names and units
        regex = re.compile(r"(?P<coord>.*)\_(?P<minmax>min|max) \[(?P<units>.*)\]")
        quantities = {}
        for colname in self._index.columns:
            if colname == "filename":
                continue

            m = regex.match(colname)
            units = m.group("units")
            magnitude = self._index[colname].values
            quantities[f"{m.group('coord')}_{m.group('minmax')}"] = ureg.Quantity(
                magnitude, units
            )

        # Populate bin bounds
        self._bins["wl"] = np.concatenate(
            (quantities["wl_min"], [quantities["wl_max"][-1]])
        )
        self._bins["wn"] = np.concatenate(
            (quantities["wn_max"], [quantities["wn_min"][-1]])
        )

    def __repr__(self) -> str:
        result = (
            f"<{type(self).__name__}> {self._dirpath}\n"
            "Index:\n"
            f"{textwrap.indent(repr(self._index), '    ')}"
        )
        return result

    @classmethod
    def from_directory(cls, dirpath) -> Aggregate:
        dirpath = Path(dirpath).resolve()

        index = (
            pd.read_csv(os.path.join(dirpath, "index.csv"))
            .sort_values(by="wl_min [nm]")
            .reset_index(drop=True)
        )

        try:
            with open(os.path.join(dirpath, "metadata.json")) as f:
                metadata = json.load(f)
        except FileNotFoundError:
            metadata = {}

        return cls(dirpath, index, metadata)

    @lru_cache
    def load_dataset(self, path) -> xr.Dataset:
        print(f"Loading {path}")
        return xr.load_dataset(path)

    def lookup_filenames(self, /, **kwargs) -> str:
        if len(kwargs) != 1:
            raise ValueError
        mode, values = next(iter(kwargs.items()))
        values = np.atleast_1d(values)
        bins = self._bins[mode]
        out_bound = (values < bins.min()) | (values > bins.max())
        if np.any(out_bound):
            # TODO: handle this error
            raise RuntimeError

        indexes = np.digitize(values.m_as(bins.u), bins=bins.m) - 1
        return list(self._index["filename"].iloc[indexes])

    def lookup_datasets(self, /, **kwargs) -> xr.Dataset:
        filenames = self.lookup_filenames(**kwargs)
        return [self.load_dataset(self._dirpath / filename) for filename in filenames]

In [2]:
agg = Aggregate.from_directory(
    "../resources/data/stable/spectra/absorption/ckd/monotropa/"
)

agg.lookup_datasets(wn=[10000.0, 11000.0] * (1.0 / ureg.cm))
agg.lookup_datasets(wl=[500.0, 600.0] * ureg.nm)

print(agg.lookup_datasets(wl=[500.0, 600.0] * ureg.nm))
print(agg)

Loading C:\Users\leroyv\Documents\src\rayference\eradiate\resources\data\stable\spectra\absorption\ckd\monotropa\monotropa-10000_10100.nc
Loading C:\Users\leroyv\Documents\src\rayference\eradiate\resources\data\stable\spectra\absorption\ckd\monotropa\monotropa-11000_11100.nc
Loading C:\Users\leroyv\Documents\src\rayference\eradiate\resources\data\stable\spectra\absorption\ckd\monotropa\monotropa-19900_20000.nc
Loading C:\Users\leroyv\Documents\src\rayference\eradiate\resources\data\stable\spectra\absorption\ckd\monotropa\monotropa-16600_16700.nc
[<xarray.Dataset> Size: 263kB
Dimensions:  (x_H2O: 4, x_O3: 2, p: 16, t: 8, w: 1, g: 32, wbv: 2, ng: 32)
Coordinates:
  * x_H2O    (x_H2O) float32 16B 0.0 0.0001 0.01 1.0
  * x_O3     (x_O3) float32 8B 0.0 1e-05
  * p        (p) float32 64B 100.0 159.0 252.8 ... 4.153e+04 6.604e+04 1.05e+05
  * t        (t) float32 32B 200.0 220.0 240.0 260.0 280.0 300.0 320.0 340.0
  * w        (w) float32 4B 501.3
  * wbv      (wbv) <U5 40B 'lower' 'upper'
  